# 02 · Selección de variables y universo de modelado

Construye una primera matriz municipal evitando identificadores, años, magnitudes brutas redundantes y fuentes con cobertura insuficiente.

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

pd.set_option('display.max_rows', 150)
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_PATH = ROOT / 'data' / 'curated' / 'municipios.parquet'
MODELS_DIR = ROOT / 'outputs' / 'models'
REPORTS_DIR = ROOT / 'outputs' / 'reports'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(DATA_PATH)
print(df.shape)

(8132, 114)


In [7]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Mostraremos hasta 150 columnas y 150 filas
pd.set_option('display.max_columns', 150)
pd.set_option('display.max_rows', 150)

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_PATH = ROOT / 'data' / 'curated' / 'municipios.parquet'
REPORTS_DIR = ROOT / 'outputs' / 'reports'
FIGURES_DIR = ROOT / 'outputs' / 'figures'
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_PATH.is_file():
    raise FileNotFoundError(f'No existe el dataset curated: {DATA_PATH}')

# Leemos el dataset de municipios
df = pd.read_parquet(DATA_PATH)
print(df.shape)

(8132, 114)


## Familias de variables

La propuesta inicial prioriza ratios, porcentajes y cambios temporales. Se incorporan las fuentes de alquiler y viviendas mediante indicadores relativos y comparables entre municipios. La criminalidad queda fuera del modelo principal por su cobertura limitada.

In [3]:
IDENTIFICADORES = ['codigo_ine', 'municipio', 'provincia']

FAMILIAS = {
    'territorio': [
        'densidad_poblacion_hab_km2',
        'distancia_costa_km',
        'porcentaje_superficie_urbana',
    ],
    'demografia': [
        'porcentaje_poblacion_menor_18',
        'porcentaje_poblacion_65_mas',
        'variacion_poblacion_1_anio',
        'variacion_poblacion_3_anios',
        'variacion_poblacion_5_anios',
        'tamanio_medio_hogar',
    ],
    'socioeconomia': [
        'renta_neta_media_persona_eur',
        'paro_por_1000_habitantes',
        'saldo_interior_por_1000_habitantes',
        'saldo_exterior_por_1000_habitantes',
    ],
    'vivienda': [
        'porcentaje_viviendas_vacias',
        'porcentaje_viviendas_uso_esporadico',
    ],
    'alquiler': [
        'alquiler_colectiva_aproximado_eur_m2',
        'alquiler_unifamiliar_aproximado_eur_m2',
        'esfuerzo_alquiler_colectiva_pct',
        'esfuerzo_alquiler_unifamiliar_pct',
    ],
    'turismo': [
        'viviendas_turisticas_por_1000_habitantes',
        'porcentaje_viviendas_turisticas_sobre_inmuebles_residenciales',
    ],
    'catastro': [
        'inmuebles_residenciales_por_1000_habitantes',
        'valor_catastral_suelo_eur_m2',
        'valor_catastral_residencial_medio_eur',
        'variacion_inmuebles_residenciales_5_anios',
        'variacion_valor_catastral_residencial_medio_eur_5_anios',
        'diferencia_porcentaje_superficie_urbana_5_anios',
        'variacion_valor_catastral_suelo_eur_m2_5_anios',
    ],
    'mercado': [
        'transacciones_por_1000_inmuebles_residenciales',
        'diferencia_transacciones_1_anio_por_1000_inmuebles_residenciales',
        'diferencia_transacciones_3_anios_por_1000_inmuebles_residenciales',
        'diferencia_transacciones_5_anios_por_1000_inmuebles_residenciales',
    ],
}

VARIABLES_MODELO = [variable for variables in FAMILIAS.values() for variable in variables]
faltantes = sorted(set(VARIABLES_MODELO) - set(df.columns))
if faltantes:
    raise ValueError(f'Faltan variables esperadas: {faltantes}')

pd.DataFrame(
    [(familia, variable) for familia, variables in FAMILIAS.items() for variable in variables],
    columns=['familia', 'variable'],
)

,familia,variable
0,territorio,densidad_poblacion_hab_km2
1,territorio,distancia_costa_km
2,territorio,porcentaje_superficie_urbana
3,demografia,porcentaje_poblacion_menor_18
4,demografia,porcentaje_poblacion_65_mas
5,demografia,variacion_poblacion_1_anio
6,demografia,variacion_poblacion_3_anios
7,demografia,variacion_poblacion_5_anios
8,demografia,tamanio_medio_hogar
9,socioeconomia,renta_neta_media_persona_eur


## Cobertura conjunta y delimitación del universo

In [ ]:
cobertura_variables = pd.DataFrame({
    'nulos': df[VARIABLES_MODELO].isna().sum(),
    'porcentaje_nulos': df[VARIABLES_MODELO].isna().mean().mul(100).round(2),
}).sort_values('porcentaje_nulos', ascending=False)
display(cobertura_variables)

# La ausencia del Catastro en territorios forales es estructural: no se imputa.
mascara_catastro = df['inmuebles_residenciales'].notna()
modelo = df.loc[mascara_catastro, [*IDENTIFICADORES, 'poblacion_2025', *VARIABLES_MODELO]].copy()
print(f'Universo con Catastro: {len(modelo):,} municipios')
print(f'Municipios completos: {modelo[VARIABLES_MODELO].notna().all(axis=1).sum():,}')

## Sensibilidad al tamaño municipal

No se aplica un umbral automáticamente. Se muestran alternativas para decidirlo justificadamente.

In [ ]:
sensibilidad = []
for minimo in [0, 50, 100, 250, 500, 1_000]:
    subconjunto = modelo.loc[modelo['poblacion_2025'].ge(minimo)]
    sensibilidad.append({
        'poblacion_minima': minimo,
        'municipios': len(subconjunto),
        'completos': subconjunto[VARIABLES_MODELO].notna().all(axis=1).sum(),
    })
display(pd.DataFrame(sensibilidad))

# Cambia este parámetro tras revisar la tabla anterior.
POBLACION_MINIMA = 0
modelo = modelo.loc[modelo['poblacion_2025'].ge(POBLACION_MINIMA)].reset_index(drop=True)

## Redundancia por correlación

In [ ]:
correlacion = modelo[VARIABLES_MODELO].corr(method='spearman').abs()
triangulo_superior = correlacion.where(np.triu(np.ones(correlacion.shape), k=1).astype(bool))
pares_altos = (
    triangulo_superior.stack()
    .rename('correlacion_abs')
    .reset_index()
    .rename(columns={'level_0': 'variable_1', 'level_1': 'variable_2'})
    .query('correlacion_abs >= 0.85')
    .sort_values('correlacion_abs', ascending=False)
)
display(pares_altos)

# Revisa los pares y elimina manualmente variables redundantes si procede.
VARIABLES_EXCLUIDAS_POR_REDUNDANCIA = []
VARIABLES_FINALES = [v for v in VARIABLES_MODELO if v not in VARIABLES_EXCLUIDAS_POR_REDUNDANCIA]

## Guardar la selección para el preprocesamiento

In [ ]:
configuracion = {
    'poblacion_minima': POBLACION_MINIMA,
    'universo': 'municipios con Catastro estatal 2026',
    'identificadores': IDENTIFICADORES,
    'variables': VARIABLES_FINALES,
    'variables_excluidas_por_redundancia': VARIABLES_EXCLUIDAS_POR_REDUNDANCIA,
}

modelo[[*IDENTIFICADORES, 'poblacion_2025', *VARIABLES_FINALES]].to_parquet(
    MODELS_DIR / 'dataset_modelo.parquet', index=False
)
(MODELS_DIR / 'configuracion_variables.json').write_text(
    json.dumps(configuracion, ensure_ascii=False, indent=2), encoding='utf-8'
)
print(f'Municipios guardados: {len(modelo):,}')
print(f'Variables guardadas: {len(VARIABLES_FINALES)}')